# LFM2.5-350M From Scratch with SKaiNET (A Standalone Notebook)

This notebook builds Liquid AI's **LFM2.5-350M** — a *hybrid* language model that mixes
**ten gated short-convolution blocks** with **six grouped-query attention blocks** — entirely
from SKaiNET tensor primitives, loads the real pretrained checkpoint, proves the
implementation against pinned reference traces, and chats with it.

Unlike a pure-attention transformer, the model's per-request memory is not one KV cache:
each convolution layer keeps a tiny rolling **history window**, each attention layer keeps a
**KV cache**, and one position counter ties them together. Making that hybrid state explicit —
and showing that chunked, cached execution matches a full forward pass — is the point of this
notebook.

The implementation is a faithful, cell-by-cell transcription of the externally qualified
LFM2.5 evaluation on SKaiNET (host JVM and Arduino VENTUNO Q): the eager model it derives
from reproduced 4,924,416 reference logit/token comparisons against the pinned
Hugging Face Transformers reference. Where that evaluation made a numerical policy decision
(for example *balanced* FP32 RMS reductions), this notebook keeps the decision and explains it.

- **Model code license:** MIT (this notebook and the SKaiNET stack).
- **Weights:** [`LiquidAI/LFM2.5-350M`](https://huggingface.co/LiquidAI/LFM2.5-350M) under the
  **LFM Open License v1.0** — fetched separately, never bundled.
- Training, quantization, GPU/NPU execution and cartridge packaging are out of scope here;
  the closing section links to those tracks.

In [1]:
// The exact engine version the LFM2.5 evaluation qualified: SKaiNET 0.62.0 (JVM CPU).
@file:DependsOn("sk.ainet.core:skainet-lang-core-jvm:0.62.0")
@file:DependsOn("sk.ainet.core:skainet-backend-api-jvm:0.62.0")
@file:DependsOn("sk.ainet.core:skainet-backend-cpu-jvm:0.62.0")
@file:DependsOn("sk.ainet.core:skainet-frontend-core-jvm:0.62.0")
@file:DependsOn("sk.ainet.core:skainet-frontend-text-jvm:0.62.0")
@file:DependsOn("sk.ainet.core:skainet-io-core-jvm:0.62.0")
@file:DependsOn("sk.ainet.core:skainet-io-safetensors-jvm:0.62.0")
@file:DependsOn("org.jetbrains.kotlinx:kotlinx-coroutines-core:1.11.0")
@file:DependsOn("org.jetbrains.kotlinx:kotlinx-serialization-json:1.9.0")

Tested with the Kotlin Jupyter kernel on **JDK 21**. Recommended kernel JVM options
(`KOTLIN_JUPYTER_JAVA_OPTS`): `-Xmx10g --add-modules=jdk.incubator.vector` — the heap bound
covers the FP32 parameters (≈1.42 GB) plus working tensors, and the incubator module lets the
SKaiNET CPU backend use SIMD. The notebook holds **one** copy of the real parameters; the tiny
demonstration models use their own throwaway tensors.

In [2]:
import kotlinx.coroutines.CoroutineScope
import kotlinx.coroutines.Dispatchers
import kotlinx.coroutines.future.future
import kotlinx.serialization.json.*
import sk.ainet.context.DirectCpuExecutionContext
import sk.ainet.context.ExecutionContext
import sk.ainet.frontend.tokenizer.TextFrontEnds
import sk.ainet.io.JvmRandomAccessSource
import sk.ainet.io.safetensors.SafeTensorsParametersLoader
import sk.ainet.io.safetensors.StreamingSafeTensorsReader
import sk.ainet.lang.tensor.Shape
import sk.ainet.lang.tensor.Tensor
import sk.ainet.lang.types.FP32
import java.io.File
import java.security.MessageDigest
import kotlin.math.abs
import kotlin.math.cos
import kotlin.math.exp
import kotlin.math.pow
import kotlin.math.sin
import kotlin.math.sqrt
import kotlin.random.Random
import kotlin.system.measureTimeMillis

// Everything in this model is FP32 — one alias keeps the signatures readable.
typealias FTensor = Tensor<FP32, Float>

// One ExecutionContext per inference session; it owns the tensor factory and CPU schedule.
val ctx = DirectCpuExecutionContext.create()

// Tensor construction helper: explicit shape + row-major FloatArray, nothing hidden.
fun ft(shape: Shape, values: FloatArray): FTensor =
    ctx.fromFloatArray<FP32, Float>(shape, FP32::class, values)

// Bridge for the suspend safetensors loader. Not runBlocking: the notebook kernel bundles an
// older kotlinx-coroutines than SKaiNET's 1.11, and future {}.get() is stable across both.
fun <T> blocking(block: suspend CoroutineScope.() -> T): T =
    CoroutineScope(Dispatchers.IO).future { block() }.get()

fun sha256(bytes: ByteArray): String =
    MessageDigest.getInstance("SHA-256").digest(bytes).joinToString("") { "%02x".format(it) }

fun sha256(file: File): String {
    val digest = MessageDigest.getInstance("SHA-256")
    file.inputStream().buffered().use { input ->
        val buffer = ByteArray(1 shl 20)
        while (true) {
            val count = input.read(buffer)
            if (count < 0) break
            digest.update(buffer, 0, count)
        }
    }
    return digest.digest().joinToString("") { "%02x".format(it) }
}

println("JVM " + System.getProperty("java.version") + " on " +
    System.getProperty("os.name") + "/" + System.getProperty("os.arch"))

[SKaiNET] Using SIMD-accelerated CPU operations (Vector API)
JVM 21.0.10 on Mac OS X/aarch64


## 1 · Model identity

Everything below is pinned to one immutable checkpoint revision, and every file is verified
by SHA-256 before it is interpreted. Downloads stay **outside** the notebook: fetch the five
metadata files and the weights once, from the exact revision, then point `LFM2_MODEL_DIR`
at that directory.

```sh
hf download LiquidAI/LFM2.5-350M --revision 9e6c6ccf47cd318696e137d381a7ded8fe4df09f \
  config.json generation_config.json tokenizer_config.json chat_template.jinja \
  tokenizer.json model.safetensors --local-dir /path/to/LFM2.5-350M
export LFM2_MODEL_DIR=/path/to/LFM2.5-350M
```

Accepting the LFM Open License v1.0 on the Hugging Face model page is part of that step.

In [3]:
val MODEL_REPOSITORY = "LiquidAI/LFM2.5-350M"
val MODEL_REVISION = "9e6c6ccf47cd318696e137d381a7ded8fe4df09f"

val modelDir = File(requireNotNull(System.getenv("LFM2_MODEL_DIR")) {
    "Set LFM2_MODEL_DIR to a local directory with the pinned $MODEL_REPOSITORY files"
})
require(modelDir.isDirectory) { "LFM2_MODEL_DIR is not a directory: $modelDir" }

// SHA-256 of every file this notebook reads, at revision 9e6c6ccf.
val filePins = linkedMapOf(
    "config.json" to "720b43d6ddc2ed25be23eed355aefcf342434a176dedad23dbe0a5e3ac24bbb8",
    "generation_config.json" to "0ec7d228c4de7e3453e98bf6910c7eb7841e87fede63ea11c0f75a5c64e1c3c3",
    "tokenizer_config.json" to "3701f370c70034d06e28947ff51c1063983393319f429e3ce58f06a45fae67cc",
    "chat_template.jinja" to "ba551d58630afa3190b1be3602e28301f3d2e9bbac978dfc49d6d825171648b6",
    "tokenizer.json" to "df1d8d5ec5d091b460562ffd545e4a5e91d17d4a0db7ebe733be34ed374377bd",
    "model.safetensors" to "1c9c77a4471a7f590f85240f74ed1fc26df7fbde88c3006724e2f93ca993ea4e",
)
for ((name, pin) in filePins) {
    val file = File(modelDir, name)
    require(file.isFile) { "Missing $name under $modelDir" }
    val actual = sha256(file)
    require(actual == pin) { "SHA-256 mismatch for $name: $actual" }
    println("verified  %-24s %,13d bytes  %s…".format(name, file.length(), actual.take(16)))
}

verified  config.json                      1,281 bytes  720b43d6ddc2ed25…


verified  generation_config.json             134 bytes  0ec7d228c4de7e34…


verified  tokenizer_config.json              595 bytes  3701f370c70034d0…


verified  chat_template.jinja              5,487 bytes  ba551d58630afa31…


verified  tokenizer.json               4,733,389 bytes  df1d8d5ec5d091b4…


verified  model.safetensors          708,984,464 bytes  1c9c77a4471a7f59…


## 2 · Architecture

LFM2.5-350M is a 16-layer decoder. Each layer is either a **short-convolution block** or a
**full-attention block**, in checkpoint order:

```
layer    0    1    2    3    4    5    6    7    8    9   10   11   12   13   14   15
kind   conv conv attn conv conv attn conv conv attn conv attn conv attn conv attn conv
```

| dimension | value |
| --- | --- |
| hidden width | 1024 |
| vocabulary (padded) | 65,536 |
| query heads / KV heads | 16 / 8 (head width 64) |
| feed-forward width | 4608 (derived below) |
| convolution window | 3 |
| RoPE θ | 1,000,000 |
| declared context | 128,000 — this notebook allocates a **tested** 256 |

Both block kinds share the same skeleton — `x + Mixer(RMSNorm(x))` then
`y + SwiGLU(RMSNorm(y))` — they differ only in the mixer and in what it remembers.
The feed-forward width is not stored directly: the checkpoint says *start from 6656, take
two thirds, round up to a multiple of 256*, which the next cell computes and pins.

In [4]:
val configRoot = Json.parseToJsonElement(File(modelDir, "config.json").readText()).jsonObject
fun cfgInt(name: String): Int = configRoot.getValue(name).jsonPrimitive.int

// Fail closed on anything this implementation does not cover.
require(configRoot.getValue("model_type").jsonPrimitive.content == "lfm2")
require(configRoot.getValue("architectures").jsonArray.map { it.jsonPrimitive.content } ==
    listOf("Lfm2ForCausalLM"))
require(configRoot.getValue("tie_embedding").jsonPrimitive.boolean) { "Tied output head expected" }
require(configRoot.getValue("block_use_swiglu").jsonPrimitive.boolean)
require(!configRoot.getValue("conv_bias").jsonPrimitive.boolean)

val layerTypes = configRoot.getValue("layer_types").jsonArray.map { it.jsonPrimitive.content }
require(layerTypes.size == cfgInt("num_hidden_layers") &&
    layerTypes.all { it in setOf("conv", "full_attention") })

// Feed-forward sizing, exactly as the reference computes it: 6656 → ⌊2·6656/3⌋ = 4437 →
// ×1.0 → round up to a multiple of 256 → 4608.
require(configRoot.getValue("block_auto_adjust_ff_dim").jsonPrimitive.boolean)
val rawFf = cfgInt("block_ff_dim")
val multiple = cfgInt("block_multiple_of")
val multiplier = configRoot.getValue("block_ffn_dim_multiplier").jsonPrimitive.double
val scaled = (multiplier * (2L * rawFf / 3L)).toLong()
val intermediate = (multiple * ((scaled + multiple - 1) / multiple)).toInt()
require(intermediate == 4608) { "Unexpected feed-forward width $intermediate" }

val rope = configRoot.getValue("rope_parameters").jsonObject
require(rope.getValue("rope_type").jsonPrimitive.content == "default")

val generationRoot =
    Json.parseToJsonElement(File(modelDir, "generation_config.json").readText()).jsonObject

data class Lfm2Dims(
    val width: Int, val vocabulary: Int, val queryHeads: Int, val kvHeads: Int,
    val headDim: Int, val intermediate: Int, val kernel: Int, val epsilon: Float,
    val theta: Double, val layerTypes: List<String>, val contextLimit: Int,
    val bosTokenId: Int, val eosTokenIds: Set<Int>,
)

val dims = Lfm2Dims(
    width = cfgInt("hidden_size"),
    vocabulary = cfgInt("vocab_size"),
    queryHeads = cfgInt("num_attention_heads"),
    kvHeads = cfgInt("num_key_value_heads"),
    headDim = cfgInt("hidden_size") / cfgInt("num_attention_heads"),
    intermediate = intermediate,
    kernel = cfgInt("conv_L_cache"),
    epsilon = configRoot.getValue("norm_eps").jsonPrimitive.float,
    theta = rope.getValue("rope_theta").jsonPrimitive.double,
    layerTypes = layerTypes,
    contextLimit = 256, // tested allocation bound, far below the advertised 128k
    bosTokenId = generationRoot.getValue("bos_token_id").jsonPrimitive.int,
    eosTokenIds = setOf(generationRoot.getValue("eos_token_id").jsonPrimitive.int),
)
require(dims.width == dims.queryHeads * dims.headDim && dims.queryHeads % dims.kvHeads == 0)
require(dims.headDim % 2 == 0) { "Split-half RoPE requires an even head width" }
require(dims.contextLimit <= cfgInt("max_position_embeddings"))

println("width=${dims.width} vocab=${dims.vocabulary} heads=${dims.queryHeads}/${dims.kvHeads}×${dims.headDim}")
println("ffn=${dims.intermediate} kernel=${dims.kernel} eps=${dims.epsilon} theta=${dims.theta}")
println("schedule: " + dims.layerTypes.withIndex()
    .joinToString(" ") { (i, k) -> "$i:" + if (k == "conv") "C" else "A" })
println("BOS=${dims.bosTokenId} EOS=${dims.eosTokenIds}")

width=1024 vocab=65536 heads=16/8×64
ffn=4608 kernel=3 eps=1.0E-5 theta=1000000.0


schedule: 0:C 1:C 2:A 

3:C 4:C 5:A 6:C 7:C 8:A 9:C 10:A 11:C 12:A 13:C 14:A 15:C
BOS=1 EOS=[7]


## 3 · RMS normalization — and a real numerical lesson

Every norm in LFM2 is a plain RMSNorm with a learned scale:
$\mathrm{rms}(x) = x \cdot \big(\tfrac{1}{d}\textstyle\sum_i x_i^2 + \varepsilon\big)^{-1/2} \cdot w$

The interesting part is **how the sum is computed**. The first full-model run of the LFM2.5
evaluation used one serial FP32 reduction over the 1024-wide rows — and *failed* its reference
comparison: in wide single-precision sums, rounding error accumulates with sequence position
until greedy decoding picks different tokens. Splitting the mean into two shorter reductions
(`[rows, width] → [rows, width/g, g]`, mean over each axis) keeps every partial sum short, and
that **BALANCED_FP32** policy is what the accepted evaluation ran everywhere. The failure and
its correction are retained in the evaluation protocol; this notebook keeps the balanced
policy as the default and keeps the serial variant available for comparison.

In [5]:
enum class RmsReductionPolicy { SERIAL_FP32, BALANCED_FP32 }

/** FP32 throughout. Two shorter reductions limit rounding drift in wide RMS sums. */
fun rmsNormalize(
    input: FTensor, weight: FTensor, ctx: ExecutionContext,
    policy: RmsReductionPolicy = RmsReductionPolicy.BALANCED_FP32,
    epsilon: Float = 1e-5f,
): FTensor {
    val ops = ctx.ops
    val rows = input.shape[0]
    val width = input.shape[1]
    require(input.shape.dimensions.size == 2 && weight.shape == Shape(width))
    val squared = ops.multiply(input, input)
    var factor = 1
    if (policy == RmsReductionPolicy.BALANCED_FP32) {
        while (factor * 2 <= width / (factor * 2) && width % (factor * 2) == 0) factor *= 2
    }
    val mean = if (factor == 1) ops.mean(squared, dim = 1) else {
        val grouped = ops.reshape(squared, Shape(rows, width / factor, factor))
        ops.mean(ops.mean(grouped, dim = 2), dim = 1)
    }
    val inverse = ops.rdivScalar(1f, ops.sqrt(ops.addScalar(mean, epsilon)))
    return ops.multiply(weight, ops.multiply(input, ops.reshape(inverse, Shape(rows, 1))))
}

// Tiny check against an independent double-precision reference: on an 8-wide row both
// policies agree with it to float precision — the policies only diverge in wide sums.
val demoInput = ft(Shape(1, 8), FloatArray(8) { (it + 1).toFloat() })
val demoWeight = ft(Shape(8), FloatArray(8) { 1f })
val serial = rmsNormalize(demoInput, demoWeight, ctx, RmsReductionPolicy.SERIAL_FP32)
    .data.copyToFloatArray()
val balanced = rmsNormalize(demoInput, demoWeight, ctx).data.copyToFloatArray()
val scale = 1.0 / sqrt((1..8).sumOf { (it * it).toDouble() } / 8 + 1e-5)
for (i in 0 until 8) {
    val reference = ((i + 1) * scale).toFloat()
    check(abs(serial[i] - reference) < 1e-6f) { "serial[$i]" }
    check(abs(balanced[i] - reference) < 1e-6f) { "balanced[$i]" }
}
println("serial   : " + serial.joinToString(" ") { "%.6f".format(it) })
println("balanced : " + balanced.joinToString(" ") { "%.6f".format(it) })
println("Both match the scalar double-precision reference within 1e-6.")

serial   : 0.198029 0.396059 0.594088 0.792118 0.990147 1.188177 1.386206 1.584236


balanced : 0.198029 0.396059 0.594088 0.792118 0.990147 1.188177 1.386206 1.584236


Both match the scalar double-precision reference within 1e-6.


## 4 · The gated short convolution

The convolution mixer is LFM2's replacement for attention in ten of the sixteen layers.
For input $x \in [T, D]$:

1. `in_proj` lifts $x$ to $[T, 3D]$ and splits it into three lanes $B, C, v$;
2. the **gated value** $B \odot v$ runs through a *depthwise causal* convolution of width
   $K{=}3$ over time (each channel has its own 3-tap filter);
3. the result is gated again by $C$ and projected back by `out_proj`.

A token therefore sees exactly the $K{-}1$ previous time steps — which is why the entire
recurrent state of a conv layer is one tiny $[D, K]$ **history window** of the most recent
gated values. Chunked generation works by sliding that window forward; no growing cache,
no position dependence.

In [6]:
class ShortConvOp(
    val width: Int,
    val kernel: Int,
    private val inWeight: FTensor,   // [3D, D] — lanes B, C, v stacked
    private val convWeight: FTensor, // [D, 1, K] — one causal filter per channel
    private val outWeight: FTensor,  // [D, D]
) {
    init {
        require(inWeight.shape == Shape(3 * width, width))
        require(convWeight.shape == Shape(width, 1, kernel))
        require(outWeight.shape == Shape(width, width))
    }

    /** Input [T,D], history [D,K] → output [T,D] and the next [D,K] history. */
    fun forward(input: FTensor, history: FTensor, ctx: ExecutionContext): Pair<FTensor, FTensor> {
        require(input.shape.dimensions.size == 2 && input.shape[0] > 0 && input.shape[1] == width)
        require(history.shape == Shape(width, kernel))
        val length = input.shape[0]
        val ops = ctx.ops
        val projected = ops.matmulWeightTransposed(input, inWeight)
        val (b, c, values) = ops.split(projected, width, 1)
        val gated = ops.multiply(b, values)
        // Prepend the history, channel-major: [D, K + T].
        val combined = ops.concat(listOf(history, ops.transpose(gated)), dim = 1)
        // K history samples are retained; only the most recent K-1 affect a new token.
        val window = ops.narrow(combined, dim = 1, start = 1, length = kernel - 1 + length)
        val convolved = ops.conv1d(
            ops.reshape(window, Shape(1, width, kernel - 1 + length)),
            convWeight, bias = null, padding = 0, groups = width,
        )
        val timeMajor = ops.transpose(ops.reshape(convolved, Shape(width, length)))
        val output = ops.matmulWeightTransposed(ops.multiply(c, timeMajor), outWeight)
        val nextHistory = ops.narrow(combined, dim = 1, start = length, length = kernel)
        return output to nextHistory
    }
}

// ── Check against an independent scalar reference, and check the state contract. ──
val cw = 2; val ck = 3
val demoRng = Random(7)
fun demoArray(size: Int) = FloatArray(size) { (demoRng.nextFloat() - 0.5f) * 2f }
val convIn = demoArray(3 * cw * cw); val convK = demoArray(cw * ck); val convOut = demoArray(cw * cw)
val conv = ShortConvOp(cw, ck,
    ft(Shape(3 * cw, cw), convIn), ft(Shape(cw, 1, ck), convK), ft(Shape(cw, cw), convOut))

val xs = demoArray(4 * cw) // four tokens
fun scalarReference(tokens: Int): Array<DoubleArray> {
    // gated[t][d] = B·x ⊙ v·x ; conv over (history=0, gated) ; gate by C·x ; out_proj
    val gated = Array(tokens) { t -> DoubleArray(cw) { d ->
        val b = (0 until cw).sumOf { j -> convIn[d * cw + j].toDouble() * xs[t * cw + j] }
        val v = (0 until cw).sumOf { j -> convIn[(2 * cw + d) * cw + j].toDouble() * xs[t * cw + j] }
        b * v
    } }
    return Array(tokens) { t -> DoubleArray(cw) { d ->
        var acc = 0.0
        for (tap in 0 until ck) {
            val source = t - (ck - 1) + tap // zero history before the first token
            if (source >= 0) acc += convK[d * ck + tap].toDouble() * gated[source][d]
        }
        val cGate = (0 until cw).sumOf { j -> convIn[(cw + d) * cw + j].toDouble() * xs[t * cw + j] }
        acc * cGate
    }.let { mixed -> DoubleArray(cw) { d ->
        (0 until cw).sumOf { j -> convOut[d * cw + j].toDouble() * mixed[j] }
    } } }
}
val zeroHistory = ft(Shape(cw, ck), FloatArray(cw * ck))
val (fullOut, fullHistory) = conv.forward(ft(Shape(4, cw), xs), zeroHistory, ctx)
val fullValues = fullOut.data.copyToFloatArray()
val reference = scalarReference(4)
for (t in 0 until 4) for (d in 0 until cw)
    check(abs(fullValues[t * cw + d] - reference[t][d]) < 1e-5f) { "conv reference ($t,$d)" }

// Chunked (2 + 2 tokens, carrying the history) must equal the full pass bit for bit.
val (chunkA, historyA) = conv.forward(ft(Shape(2, cw), xs.copyOfRange(0, 2 * cw)), zeroHistory, ctx)
val (chunkB, historyB) = conv.forward(ft(Shape(2, cw), xs.copyOfRange(2 * cw, 4 * cw)), historyA, ctx)
val chunked = chunkA.data.copyToFloatArray() + chunkB.data.copyToFloatArray()
val convDrift = fullValues.indices.maxOf { abs(fullValues[it] - chunked[it]) }
check(convDrift < 1e-6f) { "chunked conv drift $convDrift" }
val historyFull = fullHistory.data.copyToFloatArray()
val historyChunked = historyB.data.copyToFloatArray()
check(historyFull.indices.all { abs(historyFull[it] - historyChunked[it]) < 1e-6f })
println("Short conv matches the scalar reference; chunked replay max |Δ| = $convDrift.")

Short conv matches the scalar reference; chunked replay max |Δ| = 0.0.


## 5 · The convolution block

A full layer wraps the mixer in the shared skeleton — two RMS norms, two residual
connections and a **SwiGLU** feed-forward
($W_2\,(\mathrm{SiLU}(W_1 h) \odot W_3 h)$, width 1024 → 4608 → 1024):

```
x ──RMSNorm──ShortConv──(+x)──RMSNorm──SwiGLU──(+)──▶
```

Because the only stateful part is the convolution history, a block that is handed the
history it previously emitted continues *exactly* where it stopped.

In [7]:
class ConvBlockOp(
    val width: Int, val kernel: Int, val intermediateSize: Int, val normEps: Float,
    private val convolution: ShortConvOp,
    private val operatorNormWeight: FTensor, private val ffnNormWeight: FTensor,
    private val w1: FTensor, private val w2: FTensor, private val w3: FTensor,
    val rmsReduction: RmsReductionPolicy = RmsReductionPolicy.BALANCED_FP32,
) {
    private fun rms(input: FTensor, weight: FTensor, ctx: ExecutionContext) =
        rmsNormalize(input, weight, ctx, rmsReduction, normEps)

    fun forward(input: FTensor, history: FTensor, ctx: ExecutionContext): Pair<FTensor, FTensor> {
        val ops = ctx.ops
        val (convolved, nextHistory) = convolution.forward(rms(input, operatorNormWeight, ctx), history, ctx)
        val residual = ops.add(convolved, input)
        val ffn = rms(residual, ffnNormWeight, ctx)
        val mlp = ops.matmulWeightTransposed(
            ops.multiply(ops.silu(ops.matmulWeightTransposed(ffn, w1)), ops.matmulWeightTransposed(ffn, w3)),
            w2,
        )
        return ops.add(residual, mlp) to nextHistory
    }
}

// State contract on the whole block: 1+3 chunked tokens == 4 at once.
val blockDemo = ConvBlockOp(cw, ck, 4, 1e-5f, conv,
    ft(Shape(cw), floatArrayOf(1.1f, 0.9f)), ft(Shape(cw), floatArrayOf(0.8f, 1.2f)),
    ft(Shape(4, cw), demoArray(4 * cw)), ft(Shape(cw, 4), demoArray(4 * cw)), ft(Shape(4, cw), demoArray(4 * cw)))
val (blockFull, _) = blockDemo.forward(ft(Shape(4, cw), xs), zeroHistory, ctx)
val (blockA, blockHistoryA) = blockDemo.forward(ft(Shape(1, cw), xs.copyOfRange(0, cw)), zeroHistory, ctx)
val (blockB, _) = blockDemo.forward(ft(Shape(3, cw), xs.copyOfRange(cw, 4 * cw)), blockHistoryA, ctx)
val blockChunked = blockA.data.copyToFloatArray() + blockB.data.copyToFloatArray()
val blockReference = blockFull.data.copyToFloatArray()
val blockDrift = blockReference.indices.maxOf { abs(blockReference[it] - blockChunked[it]) }
check(blockDrift < 1e-6f) { "conv block chunked replay drift $blockDrift" }
println("Conv block chunked replay max |Δ| = $blockDrift")

Conv block chunked replay max |Δ| = 0.0


## 6 · Grouped-query attention, the LFM2 way

The six attention layers use the same block skeleton with a different mixer. Three details
are specific to LFM2 and all three are part of the qualified state contract:

- **Per-head Q/K RMS norms** (width 64) are applied to the *raw* projections, before rotation.
- **Split-half RoPE**: the head width is split in halves, `rotated = [-x₂ | x₁]`,
  `out = x·cos + rotated·sin`, with the cos/sin tables carrying duplicated half-tables.
- **The cache stores normalized, rotated keys.** A cached key is never touched again —
  concatenate and attend. (Compare the Gemma notebook, which caches *unrotated* keys and
  rotates after concatenation: both are valid designs, but replaying one checkpoint's
  reference traces requires implementing *its* contract, not a plausible one.)

With 16 query heads over 8 KV heads, each KV head serves two query heads; the per-request
state of an attention layer is one `[KV, positions, 64]` pair of tensors. The causal mask and
positions enter as explicit runtime tensors, never baked into the graph.

In [8]:
// cos/sin tables for absolute positions, in the duplicated split-half layout: [T, H].
fun ropeTables(positions: IntArray, headDim: Int, theta: Double): Pair<FTensor, FTensor> {
    val half = headDim / 2
    val cosines = FloatArray(positions.size * headDim)
    val sines = FloatArray(positions.size * headDim)
    for ((row, position) in positions.withIndex()) for (index in 0 until half) {
        val angle = position * theta.pow(-2.0 * index / headDim)
        val c = cos(angle).toFloat(); val s = sin(angle).toFloat()
        cosines[row * headDim + index] = c; cosines[row * headDim + index + half] = c
        sines[row * headDim + index] = s; sines[row * headDim + index + half] = s
    }
    return ft(Shape(positions.size, headDim), cosines) to ft(Shape(positions.size, headDim), sines)
}

// Additive causal mask [T, P+T]: token t sees positions 0..P+t.
fun causalMask(past: Int, length: Int): FTensor =
    ft(Shape(length, past + length), FloatArray(length * (past + length)) { index ->
        if (index % (past + length) <= past + index / (past + length)) 0f else -Float.MAX_VALUE
    })

class AttentionBlockOp(
    val width: Int, val queryHeads: Int, val kvHeads: Int, val headDim: Int,
    val intermediateSize: Int, val normEps: Float,
    private val operatorNorm: FTensor, private val ffnNorm: FTensor,
    private val qWeight: FTensor, private val kWeight: FTensor,
    private val vWeight: FTensor, private val outWeight: FTensor,
    private val qNorm: FTensor, private val kNorm: FTensor,
    private val w1: FTensor, private val w2: FTensor, private val w3: FTensor,
    val rmsReduction: RmsReductionPolicy = RmsReductionPolicy.BALANCED_FP32,
) {
    init {
        require(width == queryHeads * headDim && queryHeads % kvHeads == 0 && headDim % 2 == 0)
    }

    private fun rms(input: FTensor, weight: FTensor, ctx: ExecutionContext) =
        rmsNormalize(input, weight, ctx, rmsReduction, normEps)

    private fun rotate(input: FTensor, cos: FTensor, sin: FTensor, ctx: ExecutionContext): FTensor {
        val ops = ctx.ops
        val first = ops.narrow(input, dim = 2, start = 0, length = headDim / 2)
        val second = ops.narrow(input, dim = 2, start = headDim / 2, length = headDim / 2)
        val rotated = ops.concat(listOf(ops.mulScalar(second, -1f), first), dim = 2)
        return ops.add(ops.multiply(input, cos), ops.multiply(rotated, sin))
    }

    private fun repeatKv(input: FTensor, ctx: ExecutionContext): FTensor {
        if (queryHeads == kvHeads) return input
        val groups = queryHeads / kvHeads
        return ctx.ops.concat((0 until queryHeads).map { head ->
            ctx.ops.narrow(input, dim = 0, start = head / groups, length = 1)
        }, dim = 0)
    }

    /** Input [T,D], past KV [KV,P,H] or null, cos/sin [T,H], mask [T,P+T]. */
    fun forward(
        input: FTensor, pastKey: FTensor?, pastValue: FTensor?,
        cos: FTensor, sin: FTensor, mask: FTensor, ctx: ExecutionContext,
    ): Triple<FTensor, FTensor, FTensor> {
        require((pastKey == null) == (pastValue == null))
        val length = input.shape[0]
        val past = pastKey?.shape?.get(1) ?: 0
        require(mask.shape == Shape(length, past + length))
        val ops = ctx.ops
        val normalized = rms(input, operatorNorm, ctx)
        // Per-head RMS before rotation, then permute to head-major [heads, T, H].
        val queryNormalized = ops.reshape(
            rms(ops.reshape(ops.matmulWeightTransposed(normalized, qWeight),
                Shape(length * queryHeads, headDim)), qNorm, ctx),
            Shape(length, queryHeads, headDim))
        val keyNormalized = ops.reshape(
            rms(ops.reshape(ops.matmulWeightTransposed(normalized, kWeight),
                Shape(length * kvHeads, headDim)), kNorm, ctx),
            Shape(length, kvHeads, headDim))
        val query = rotate(ops.permute(queryNormalized, intArrayOf(1, 0, 2)), cos, sin, ctx)
        val currentKey = rotate(ops.permute(keyNormalized, intArrayOf(1, 0, 2)), cos, sin, ctx)
        val currentValue = ops.permute(
            ops.reshape(ops.matmulWeightTransposed(normalized, vWeight), Shape(length, kvHeads, headDim)),
            intArrayOf(1, 0, 2))
        // The cache holds exactly these: normalized, rotated keys and raw values.
        val key = if (pastKey == null) currentKey else ops.concat(listOf(pastKey, currentKey), dim = 1)
        val value = if (pastValue == null) currentValue else ops.concat(listOf(pastValue, currentValue), dim = 1)
        val products = ops.matmul(query, ops.permute(repeatKv(key, ctx), intArrayOf(0, 2, 1)))
        val scores = ops.add(ops.mulScalar(products, (1.0 / sqrt(headDim.toDouble())).toFloat()), mask)
        val probabilities = ops.softmax(scores, dim = 2)
        val context = ops.matmul(probabilities, repeatKv(value, ctx))
        val attended = ops.reshape(ops.permute(context, intArrayOf(1, 0, 2)), Shape(length, width))
        val residual = ops.add(ops.matmulWeightTransposed(attended, outWeight), input)
        val ffn = rms(residual, ffnNorm, ctx)
        val mlp = ops.matmulWeightTransposed(
            ops.multiply(ops.silu(ops.matmulWeightTransposed(ffn, w1)), ops.matmulWeightTransposed(ffn, w3)),
            w2,
        )
        return Triple(ops.add(residual, mlp), key, value)
    }
}
println("AttentionBlockOp defined")

AttentionBlockOp defined


In [9]:
// Tiny attention block: streaming must equal the batched pass, and causality must hold.
val aw = 8; val aq = 4; val akv = 2; val ah = 2
val attnDemo = AttentionBlockOp(aw, aq, akv, ah, 4, 1e-5f,
    ft(Shape(aw), demoArray(aw)), ft(Shape(aw), demoArray(aw)),
    ft(Shape(aq * ah, aw), demoArray(aq * ah * aw)), ft(Shape(akv * ah, aw), demoArray(akv * ah * aw)),
    ft(Shape(akv * ah, aw), demoArray(akv * ah * aw)), ft(Shape(aw, aq * ah), demoArray(aw * aq * ah)),
    ft(Shape(ah), floatArrayOf(1.2f, 0.8f)), ft(Shape(ah), floatArrayOf(0.9f, 1.1f)),
    ft(Shape(4, aw), demoArray(4 * aw)), ft(Shape(aw, 4), demoArray(aw * 4)), ft(Shape(4, aw), demoArray(4 * aw)))

val tokenA = demoArray(aw); val tokenB = demoArray(aw); val tokenC = demoArray(aw)
val (cosFull, sinFull) = ropeTables(intArrayOf(0, 1), ah, 1e6)
val (batched, _, _) = attnDemo.forward(ft(Shape(2, aw), tokenA + tokenB), null, null,
    cosFull, sinFull, causalMask(0, 2), ctx)
val batchedValues = batched.data.copyToFloatArray()

// Streaming: one token, then the next against the cached normalized+rotated key.
val (cos0, sin0) = ropeTables(intArrayOf(0), ah, 1e6)
val (step0, key0, value0) = attnDemo.forward(ft(Shape(1, aw), tokenA), null, null, cos0, sin0, causalMask(0, 1), ctx)
val (cos1, sin1) = ropeTables(intArrayOf(1), ah, 1e6)
val (step1, _, _) = attnDemo.forward(ft(Shape(1, aw), tokenB), key0, value0, cos1, sin1, causalMask(1, 1), ctx)
val streamed = step0.data.copyToFloatArray() + step1.data.copyToFloatArray()
val streamDrift = batchedValues.indices.maxOf { abs(batchedValues[it] - streamed[it]) }
check(streamDrift < 1e-5f) { "attention streaming drift $streamDrift" }

// Causality: replacing the SECOND token must not change the first row at all.
val (altered, _, _) = attnDemo.forward(ft(Shape(2, aw), tokenA + tokenC), null, null,
    cosFull, sinFull, causalMask(0, 2), ctx)
val alteredValues = altered.data.copyToFloatArray()
for (i in 0 until aw) check(alteredValues[i] == batchedValues[i]) { "future token leaked into position 0" }
println("Attention streaming max |Δ| = $streamDrift; causality holds.")

Attention streaming max |Δ| = 0.0; causality holds.


## 7 · Hybrid state: what the model actually remembers

One generation request owns exactly this, nothing else:

| layer kind | per-layer state | size at 256 context |
| --- | --- | --- |
| 10 × conv | history `[1024, 3]` | 12 KB each, **constant** |
| 6 × attention | keys + values `[8, P, 64]` | grows with P, ≈512 KB each at P=256 |
| — | one position counter | |

`forward(tokens, state)` consumes the state and commits the advanced state only on success;
`reset()` returns the request to position 0. Every consistency rule the qualified
implementation enforces (complete state or empty state, cache length == position) is kept —
these checks are what caught real cache bugs during the evaluation.

In [10]:
class HybridState {
    var position: Int = 0
        private set
    val convolution = mutableMapOf<Int, FTensor>() // layer → [D, K] history
    val keys = mutableMapOf<Int, FTensor>()        // layer → [KV, P, H] rotated keys
    val values = mutableMapOf<Int, FTensor>()      // layer → [KV, P, H] values

    fun reset() { position = 0; convolution.clear(); keys.clear(); values.clear() }

    fun commit(nextPosition: Int, nextConvolution: Map<Int, FTensor>,
               nextKeys: Map<Int, FTensor>, nextValues: Map<Int, FTensor>) {
        convolution.clear(); convolution.putAll(nextConvolution)
        keys.clear(); keys.putAll(nextKeys)
        values.clear(); values.putAll(nextValues)
        position = nextPosition
    }
}

/** Expected checkpoint tensors: names and layouts for one configured architecture. */
fun expectedWeights(d: Lfm2Dims): LinkedHashMap<String, Shape> = linkedMapOf<String, Shape>().apply {
    put("model.embed_tokens.weight", Shape(d.vocabulary, d.width))
    put("model.embedding_norm.weight", Shape(d.width))
    for ((index, kind) in d.layerTypes.withIndex()) {
        val prefix = "model.layers.$index."
        put(prefix + "operator_norm.weight", Shape(d.width))
        put(prefix + "ffn_norm.weight", Shape(d.width))
        put(prefix + "feed_forward.w1.weight", Shape(d.intermediate, d.width))
        put(prefix + "feed_forward.w2.weight", Shape(d.width, d.intermediate))
        put(prefix + "feed_forward.w3.weight", Shape(d.intermediate, d.width))
        if (kind == "conv") {
            put(prefix + "conv.in_proj.weight", Shape(3 * d.width, d.width))
            put(prefix + "conv.conv.weight", Shape(d.width, 1, d.kernel))
            put(prefix + "conv.out_proj.weight", Shape(d.width, d.width))
        } else {
            put(prefix + "self_attn.q_proj.weight", Shape(d.width, d.width))
            put(prefix + "self_attn.k_proj.weight", Shape(d.kvHeads * d.headDim, d.width))
            put(prefix + "self_attn.v_proj.weight", Shape(d.kvHeads * d.headDim, d.width))
            put(prefix + "self_attn.out_proj.weight", Shape(d.width, d.width))
            put(prefix + "self_attn.q_layernorm.weight", Shape(d.headDim))
            put(prefix + "self_attn.k_layernorm.weight", Shape(d.headDim))
        }
    }
}

/** The eager hybrid model: host embedding gather, explicit positions/mask, tied output head. */
class Lfm2Model(val dims: Lfm2Dims, val ctx: ExecutionContext, weights: Map<String, FTensor>) {
    private val embedding = weights.getValue("model.embed_tokens.weight")
    private val finalNorm = weights.getValue("model.embedding_norm.weight")
    private val convolution = mutableMapOf<Int, ConvBlockOp>()
    private val attention = mutableMapOf<Int, AttentionBlockOp>()

    init {
        val expected = expectedWeights(dims)
        require(weights.keys == expected.keys &&
            expected.all { (name, shape) -> weights.getValue(name).shape == shape }) {
            "Tensor names/layouts differ from the configured architecture"
        }
        for ((index, kind) in dims.layerTypes.withIndex()) {
            val prefix = "model.layers.$index."
            fun w(name: String) = weights.getValue(prefix + name + ".weight")
            if (kind == "conv") {
                convolution[index] = ConvBlockOp(dims.width, dims.kernel, dims.intermediate, dims.epsilon,
                    ShortConvOp(dims.width, dims.kernel, w("conv.in_proj"), w("conv.conv"), w("conv.out_proj")),
                    w("operator_norm"), w("ffn_norm"),
                    w("feed_forward.w1"), w("feed_forward.w2"), w("feed_forward.w3"))
            } else {
                attention[index] = AttentionBlockOp(dims.width, dims.queryHeads, dims.kvHeads,
                    dims.headDim, dims.intermediate, dims.epsilon,
                    w("operator_norm"), w("ffn_norm"),
                    w("self_attn.q_proj"), w("self_attn.k_proj"), w("self_attn.v_proj"), w("self_attn.out_proj"),
                    w("self_attn.q_layernorm"), w("self_attn.k_layernorm"),
                    w("feed_forward.w1"), w("feed_forward.w2"), w("feed_forward.w3"))
            }
        }
    }

    /** Returns logits [T, vocabulary] for this chunk and commits the advanced state. */
    fun forward(inputIds: IntArray, state: HybridState): FTensor {
        val d = dims
        require(inputIds.isNotEmpty() && inputIds.all { it in 0 until d.vocabulary })
        require(state.position.toLong() + inputIds.size <= d.contextLimit) { "Context limit exceeded" }
        val past = state.position
        require((past == 0 && state.convolution.isEmpty() && state.keys.isEmpty() && state.values.isEmpty()) ||
            (past > 0 && state.convolution.keys == convolution.keys &&
                state.keys.keys == attention.keys && state.values.keys == attention.keys)) {
            "Hybrid state is incomplete or inconsistent"
        }
        val length = inputIds.size
        // Host-side embedding gather: one row per token, no 65k×1024 matmul.
        val gathered = FloatArray(length * d.width) { index ->
            embedding.data[inputIds[index / d.width], index % d.width]
        }
        var hidden = ft(Shape(length, d.width), gathered)
        val (cos, sin) = ropeTables(IntArray(length) { past + it }, d.headDim, d.theta)
        val mask = causalMask(past, length)
        val nextConvolution = state.convolution.toMutableMap()
        val nextKeys = state.keys.toMutableMap()
        val nextValues = state.values.toMutableMap()
        val ops = ctx.ops
        for ((index, kind) in d.layerTypes.withIndex()) {
            hidden = if (kind == "conv") {
                val history = state.convolution[index]
                    ?: ft(Shape(d.width, d.kernel), FloatArray(d.width * d.kernel))
                val (output, nextHistory) = convolution.getValue(index).forward(hidden, history, ctx)
                nextConvolution[index] = nextHistory
                output
            } else {
                val key = state.keys[index]; val value = state.values[index]
                require(past == 0 || (key?.shape == Shape(d.kvHeads, past, d.headDim) &&
                    value?.shape == Shape(d.kvHeads, past, d.headDim))) { "Attention cache does not match the global position" }
                val (output, nextKey, nextValue) = attention.getValue(index)
                    .forward(hidden, key, value, cos, sin, mask, ctx)
                nextKeys[index] = nextKey; nextValues[index] = nextValue
                output
            }
        }
        val normalized = rmsNormalize(hidden, finalNorm, ctx, RmsReductionPolicy.BALANCED_FP32, d.epsilon)
        val logits = ops.matmulWeightTransposed(normalized, embedding) // tied output head
        state.commit(past + length, nextConvolution, nextKeys, nextValues)
        return logits
    }
}
println("Lfm2Model defined")

Lfm2Model defined


Before touching the real checkpoint, the same contract checks run on a **tiny random
model** (width 8, three layers `conv/attention/conv`) — cheap enough to run on every kernel
restart, and failures here point at the implementation rather than at 700 MB of weights.

In [11]:
val tinyDims = Lfm2Dims(
    width = 8, vocabulary = 32, queryHeads = 4, kvHeads = 2, headDim = 2,
    intermediate = 16, kernel = 3, epsilon = 1e-5f, theta = 1e6,
    layerTypes = listOf("conv", "full_attention", "conv"), contextLimit = 64,
    bosTokenId = 1, eosTokenIds = setOf(7),
)
val tinyRng = Random(42)
val tinyWeights = expectedWeights(tinyDims).mapValues { (_, shape) ->
    val total = shape.dimensions.fold(1) { a, b -> a * b }
    ft(shape, FloatArray(total) { (tinyRng.nextFloat() - 0.5f) * 0.4f })
}
val tiny = Lfm2Model(tinyDims, ctx, tinyWeights)

val tinyIds = intArrayOf(5, 11, 23, 2, 17)
// Full pass vs chunked (2 + 3) through committed hybrid state.
val tinyFullState = HybridState()
val tinyFull = tiny.forward(tinyIds, tinyFullState).data.copyToFloatArray()
val tinyChunkState = HybridState()
tiny.forward(tinyIds.copyOfRange(0, 2), tinyChunkState)
val tinyTail = tiny.forward(tinyIds.copyOfRange(2, 5), tinyChunkState).data.copyToFloatArray()
var tinyDrift = 0f
for (row in 0 until 3) for (column in 0 until tinyDims.vocabulary) {
    val full = tinyFull[(row + 2) * tinyDims.vocabulary + column]
    tinyDrift = maxOf(tinyDrift, abs(full - tinyTail[row * tinyDims.vocabulary + column]))
}
check(tinyDrift < 1e-5f) { "tiny model cached drift $tinyDrift" }
check(tinyFullState.position == 5 && tinyChunkState.position == 5)

// Reset really clears the request, and two sessions never share state.
tinyChunkState.reset()
check(tinyChunkState.position == 0 && tinyChunkState.keys.isEmpty())
val sessionOne = HybridState(); val sessionTwo = HybridState()
tiny.forward(intArrayOf(3, 4), sessionOne)
val freshLogits = tiny.forward(tinyIds, sessionTwo).data.copyToFloatArray()
check(freshLogits.indices.all { abs(freshLogits[it] - tinyFull[it]) < 1e-6f }) { "sessions interfered" }
println("Tiny hybrid model: cached-vs-full max |Δ| = $tinyDrift; reset and session isolation hold.")

Tiny hybrid model: cached-vs-full max |Δ| = 0.0; reset and session isolation hold.


## 8 · Loading the real checkpoint

The checkpoint stores **148 tensors in BF16** (708,984,464 bytes, already verified by
SHA-256 above). Loading them means: check the inventory against `expectedWeights` *before*
allocating anything, then stream each tensor and widen BF16 → FP32 (every BF16 value is
exactly representable in FP32, so this conversion is lossless). There is deliberately no
`lm_head.weight`: the output projection reuses the embedding matrix (`tie_embedding`).

In [12]:
val expectedReal = expectedWeights(dims)
println("expected tensors: " + expectedReal.size)

val checkpointFile = File(modelDir, "model.safetensors")
StreamingSafeTensorsReader.open(JvmRandomAccessSource.open(checkpointFile.absolutePath)).use { reader ->
    val tensors = reader.tensors
    require(tensors.size == expectedReal.size &&
        tensors.map { it.name }.toSet() == expectedReal.keys) { "Checkpoint inventory mismatch" }
    for (info in tensors) {
        val shape = expectedReal.getValue(info.name)
        require(info.shape == shape.dimensions.map(Int::toLong)) { "Shape mismatch: ${info.name}" }
        require(info.dtype == "BF16") { "Expected BF16 source tensor: ${info.name}" }
    }
}
println("inventory verified: 148 BF16 tensors, names and layouts match the architecture")

expected tensors: 148


inventory verified: 148 BF16 tensors, names and layouts match the architecture


In [13]:
val realWeights = LinkedHashMap<String, FTensor>()
val loadMillis = measureTimeMillis {
    blocking {
        SafeTensorsParametersLoader(sourceProvider = { JvmRandomAccessSource.open(checkpointFile.absolutePath) })
            .load<FP32, Float>(ctx, FP32::class) { name, tensor ->
                require(name in expectedReal && realWeights.put(name, tensor) == null) {
                    "Unexpected or repeated tensor: $name"
                }
            }
    }
}
require(realWeights.keys == expectedReal.keys) { "Incomplete checkpoint load" }
val parameterCount = expectedReal.values.sumOf { shape ->
    shape.dimensions.fold(1L) { a, b -> a * b }
}
val model = Lfm2Model(dims, ctx, realWeights)
println("loaded %,d parameters (%.2f GB as FP32) in %.1f s".format(
    parameterCount, parameterCount * 4.0 / (1 shl 30), loadMillis / 1000.0))

loaded 354,483,968 parameters (1.32 GB as FP32) in 0.2 s


## 9 · Correctness before text

Two checks run before the model is allowed near a tokenizer:

1. **Cached execution agrees with batched execution on the real weights.** The reference
   prompt below is the pinned token sequence for *"What is the capital of France?"* (its
   provenance is the next section). The last-position logits of a single 24-token pass and
   of a 16 + 8 chunked pass must agree to FP32 working precision, and must pick the same
   token.
2. The **argmax over 65,536 logits** is computed on the host from the raw array — the same
   fail-closed policy as the qualified runtime (reject non-finite logits, no hidden sampler).

The full evaluation behind this notebook compared 4,924,416 teacher-forced and greedy
positions against the pinned Hugging Face reference; these notebook cells re-establish the
end-to-end token traces from that acceptance, not the full sweep.

In [14]:
// Pinned reference prompt ids: "<|startoftext|>…What is the capital of France?…assistant\n"
val francePromptIds = intArrayOf(1, 6, 24131, 708, 40131, 21923, 523, 7, 708, 6, 6423, 708,
    3493, 856, 779, 5706, 803, 4481, 540, 7, 708, 6, 64015, 708)

fun lastRow(logits: FTensor): FloatArray {
    val vocabulary = logits.shape[1]
    val values = logits.data.copyToFloatArray()
    return values.copyOfRange((logits.shape[0] - 1) * vocabulary, logits.shape[0] * vocabulary)
}
fun argmax(values: FloatArray): Int {
    require(values.all { it.isFinite() }) { "Non-finite logits" }
    var best = 0
    for (index in 1 until values.size) if (values[index] > values[best]) best = index
    return best
}

val batchedState = HybridState()
val batchedLast = lastRow(model.forward(francePromptIds, batchedState))
val chunkedState = HybridState()
model.forward(francePromptIds.copyOfRange(0, 16), chunkedState)
val chunkedLast = lastRow(model.forward(francePromptIds.copyOfRange(16, francePromptIds.size), chunkedState))
val realDrift = batchedLast.indices.maxOf { abs(batchedLast[it] - chunkedLast[it]) }
check(argmax(batchedLast) == argmax(chunkedLast)) { "cached execution changed the greedy token" }
check(realDrift < 5e-4f) { "cached-vs-batched drift $realDrift" }
println("real model cached-vs-batched: max |Δ| = %.2e, argmax %d in both".format(realDrift, argmax(batchedLast)))

real model cached-vs-batched: max |Δ| = 4.91e-05, argmax 41677 in both


## 10 · Tokenizer and chat template

The checkpoint ships a BPE `tokenizer.json`. One portability trap is qualified and worked
around here, exactly as in the evaluation: the tokenizer's `Split` pre-tokenizer regex uses
character classes like `\p{L}` that Rust's regex engine treats as Unicode-aware, while Java
compiles them ASCII-only unless `(?U)` is set — on JVM the raw pattern silently mis-splits
non-break spaces and non-ASCII words. The fix (an in-memory `(?U)` prefix on exactly one
`Split` regex) was later contributed upstream; metadata on disk stays untouched, and the
adaptation refuses to run if the tokenizer's structure is not the pinned one.

The chat template is rendered by a small checked Kotlin function covering this checkpoint's
**text subset** (system/user/assistant, optional `<think>` handling, generation prompt).
It is *not* a Jinja engine — and the rendered bytes are asserted against the pinned reference
below, which is what makes the shortcut safe.

In [15]:
// (?U) adaptation: Rust regex character classes are Unicode-aware; Java needs explicit U.
fun unicodeTokenizerSpecification(original: String): String {
    val root = Json.parseToJsonElement(original).jsonObject
    val pre = root.getValue("pre_tokenizer").jsonObject
    require(pre.getValue("type").jsonPrimitive.content == "Sequence") { "Expected pinned Sequence pre-tokenizer" }
    var changes = 0
    val elements = pre.getValue("pretokenizers").jsonArray.map { element ->
        val item = element.jsonObject
        if (item.getValue("type").jsonPrimitive.content != "Split") element else {
            val pattern = item.getValue("pattern").jsonObject
            val regex = pattern.getValue("Regex").jsonPrimitive.content
            changes++
            JsonObject(item + ("pattern" to JsonObject(pattern + ("Regex" to JsonPrimitive("(?U)$regex")))))
        }
    }
    require(changes == 1) { "Expected exactly one Split regex in the pinned tokenizer" }
    return JsonObject(root + ("pre_tokenizer" to JsonObject(pre + ("pretokenizers" to JsonArray(elements))))).toString()
}

val tokenizerOriginal = File(modelDir, "tokenizer.json").readText()
val tokenizerConfigText = File(modelDir, "tokenizer_config.json").readText()
val tokenizer = TextFrontEnds.fromTokenizerJson(unicodeTokenizerSpecification(tokenizerOriginal), tokenizerConfigText)

val tokenizerConfig = Json.parseToJsonElement(tokenizerConfigText).jsonObject
val bosToken = tokenizerConfig.getValue("bos_token").jsonPrimitive.content
val eosToken = tokenizerConfig.getValue("eos_token").jsonPrimitive.content

// The declared special tokens (added_tokens with special=true) are skipped for display text.
val specialTokenIds = Json.parseToJsonElement(tokenizerOriginal).jsonObject
    .getValue("added_tokens").jsonArray
    .filter { it.jsonObject.getValue("special").jsonPrimitive.boolean }
    .map { it.jsonObject.getValue("id").jsonPrimitive.int }
    .toSet()

fun decodeTokens(ids: IntArray, skipSpecial: Boolean = false): String {
    val kept = if (skipSpecial) ids.filterNot { it in specialTokenIds }.toIntArray() else ids
    // Filter whole arrays before decoding so UTF-8 byte pieces join correctly.
    return if (kept.isEmpty()) "" else tokenizer.decode(kept, skipSpecialTokens = false)
}

// BOS/EOS sanity against the generation configuration.
require(tokenizer.encode(bosToken, addSpecialTokens = false).contentEquals(intArrayOf(dims.bosTokenId)))
val eosIds = tokenizer.encode(eosToken, addSpecialTokens = false)
require(eosIds.size == 1 && eosIds[0] in dims.eosTokenIds)
println("tokenizer ready: bos '$bosToken'=${dims.bosTokenId}, eos '$eosToken'=${dims.eosTokenIds}, " +
    specialTokenIds.size.toString() + " special tokens")

tokenizer ready: bos '<|startoftext|>'=1, eos '<|im_end|>'=[7], 503 special tokens


In [16]:
data class ChatMessage(val role: String, val content: String, val thinking: String? = null)

/** Python str.strip whitespace, including NEL and the four ASCII separators. */
fun pythonWhitespace(c: Char): Boolean = c in '\u0009'..'\u000d' || c in '\u001c'..'\u0020' ||
    c == '\u0085' || c == '\u00a0' || c == '\u1680' || c in '\u2000'..'\u200a' ||
    c == '\u2028' || c == '\u2029' || c == '\u202f' || c == '\u205f' || c == '\u3000'

/** Checked Kotlin rendering of the pinned template's text subset. */
fun renderChat(
    messages: List<ChatMessage>,
    addGenerationPrompt: Boolean = true,
    preserveThinking: Boolean = false,
): String {
    require(messages.isNotEmpty())
    messages.forEachIndexed { index, message ->
        require(message.role in setOf("system", "user", "assistant")) { "Unsupported role ${message.role}" }
        require(message.role != "system" || index == 0) { "Only a leading system message is supported" }
        require(message.role == "assistant" || message.thinking == null)
    }
    val remaining = if (messages.first().role == "system") messages.drop(1) else messages
    val lastAssistant = remaining.indexOfLast { it.role == "assistant" }
    return buildString {
        append(bosToken)
        if (messages.first().role == "system" && messages.first().content.isNotEmpty()) {
            append("<|im_start|>system\n"); append(messages.first().content); append("<|im_end|>\n")
        }
        remaining.forEachIndexed { index, message ->
            append("<|im_start|>"); append(message.role); append('\n')
            if (message.role == "assistant") {
                val keepThinking = preserveThinking || index == lastAssistant
                if (keepThinking && !message.thinking.isNullOrEmpty()) {
                    append("<think>"); append(message.thinking); append("</think>")
                }
                val content = if (!keepThinking && "</think>" in message.content) {
                    message.content.substringAfterLast("</think>").trim { pythonWhitespace(it) }
                } else message.content
                append(content)
            } else append(message.content)
            append("<|im_end|>\n")
        }
        if (addGenerationPrompt) append("<|im_start|>assistant\n")
    }
}

// Byte-exact rendering + token-exact encoding against the pinned reference case.
val franceMessages = listOf(
    ChatMessage("system", "Answer briefly."),
    ChatMessage("user", "What is the capital of France?"),
)
val franceRendered = renderChat(franceMessages)
require(franceRendered == "<|startoftext|><|im_start|>system\nAnswer briefly.<|im_end|>\n" +
    "<|im_start|>user\nWhat is the capital of France?<|im_end|>\n<|im_start|>assistant\n")
val franceEncoded = tokenizer.encode(franceRendered, addSpecialTokens = false)
require(franceEncoded.contentEquals(francePromptIds)) {
    "Tokenizer output differs from the pinned reference ids"
}
println("template + tokenizer reproduce the pinned reference prompt (" + franceEncoded.size + " ids)")

template + tokenizer reproduce the pinned reference prompt (24 ids)


## 11 · Greedy generation

Generation is deliberately boring: render → encode → prefill → repeat
(host argmax → forward one token). Each request owns a **fresh `HybridState`**; the emitted
text callback receives the *cumulative* decode, because a byte-level BPE can revise an
incomplete UTF-8 suffix when the next token arrives. Stop tokens come from the generation
configuration (`<|im_end|>`, id 7), and a request that reaches its token budget reports
`length` instead of `eos` — no silent truncation.

In [17]:
data class GenerationResult(
    val promptIds: IntArray, val generatedIds: IntArray,
    val text: String, val stopReason: String,
)

fun generateGreedy(
    messages: List<ChatMessage>,
    maxNewTokens: Int = 32,
    onToken: (Int, String) -> Unit = { _, _ -> },
): GenerationResult {
    val prompt = tokenizer.encode(renderChat(messages), addSpecialTokens = false)
    require(prompt.isNotEmpty() && prompt.size + maxNewTokens - 1 <= dims.contextLimit) {
        "Prompt and decode budget must fit the tested context"
    }
    val state = HybridState()
    val generated = mutableListOf<Int>()
    var text = ""
    var logits = lastRow(model.forward(prompt, state))
    var stopReason = "length"
    while (true) {
        val token = argmax(logits)
        generated.add(token)
        text = decodeTokens(generated.toIntArray(), skipSpecial = true)
        onToken(token, text)
        if (token in dims.eosTokenIds) { stopReason = "eos"; break }
        if (generated.size == maxNewTokens) break
        logits = lastRow(model.forward(intArrayOf(token), state))
    }
    return GenerationResult(prompt, generated.toIntArray(), text, stopReason)
}
println("generateGreedy defined")

generateGreedy defined


### Reference trace 1 — and the first real answer

The expected ids/bytes below are the pinned deterministic traces from the LFM2.5 evaluation,
produced by the Hugging Face Transformers 5 reference on this exact checkpoint and then
reproduced token-for-token by the qualified SKaiNET implementation on host JVM and on the
VENTUNO Q device. If any cell above had drifted — a norm policy, a cache rule, one byte of
the template — these assertions are where it would surface.

In [18]:
val franceStart = System.nanoTime()
val france = generateGreedy(franceMessages, maxNewTokens = 8)
val franceSeconds = (System.nanoTime() - franceStart) / 1e9
require(france.generatedIds.contentEquals(intArrayOf(41677, 7))) {
    "Trace 1 ids " + france.generatedIds.toList()
}
require(decodeTokens(france.generatedIds) == "Paris<|im_end|>")
require(france.stopReason == "eos")
println("Q: What is the capital of France?")
println("A: ${france.text}   [ids ${france.generatedIds.toList()}, ${france.stopReason}, " +
    "%.1f s, %.2f tok/s]".format(franceSeconds, (france.promptIds.size + france.generatedIds.size - 1) / franceSeconds))

Q: What is the capital of France?
A: Paris   [ids [41677, 7], eos, 1.8 s, 14.16 tok/s]


### Reference trace 2 — implementation fidelity ≠ model quality

A multi-turn arithmetic prompt: *1 + 1 = 2* established in context, then *"And 2 + 2?"*.
The pinned reference answer is **"3"** — wrong. The Hugging Face reference produces exactly
the same wrong token on this checkpoint, so matching it is a *success* for the
implementation and a caution about the model: a 350M-parameter model under greedy decoding
has real quality limits, and an evaluation must separate "my port is faithful" from "the
model is right". (A single-turn `2+2=` completion does yield `4` — the failure is specific
to this conversational framing.)

In [19]:
val mathMessages = listOf(
    ChatMessage("user", "What is 1 + 1?"),
    ChatMessage("assistant", "2"),
    ChatMessage("user", "And 2 + 2?"),
)
val math = generateGreedy(mathMessages, maxNewTokens = 8)
require(math.generatedIds.contentEquals(intArrayOf(528, 7))) { "Trace 2 ids " + math.generatedIds.toList() }
require(decodeTokens(math.generatedIds) == "3<|im_end|>")
println("Q: (1+1=2 in context)  And 2 + 2?")
println("A: ${math.text}   ← matches the reference byte-for-byte, and is wrong: model quality, not a port bug")

Q: (1+1=2 in context)  And 2 + 2?


A: 3   ← matches the reference byte-for-byte, and is wrong: model quality, not a port bug


### Streaming, request isolation, and a free prompt

One unpinned prompt to see streaming output, followed by a repeat of trace 1 to show that
requests are isolated: after any number of other sessions, a fresh request reproduces the
pinned ids exactly.

In [20]:
// Each event carries the token id and the CUMULATIVE decode — a byte-level BPE may
// revise an incomplete UTF-8 suffix, so appending per-token fragments would be wrong.
val colors = generateGreedy(listOf(ChatMessage("user", "Name three primary colors, comma separated.")),
    maxNewTokens = 24, onToken = { tokenId, text ->
        println("%6d → %s".format(tokenId, text))
    })
println("[${colors.generatedIds.size} tokens, ${colors.stopReason}] ${colors.text}")

// Request isolation: the pinned trace reproduces after unrelated sessions ran.
val franceAgain = generateGreedy(franceMessages, maxNewTokens = 8)
require(franceAgain.generatedIds.contentEquals(intArrayOf(41677, 7))) { "request isolation broken" }
println("trace 1 reproduced after an unrelated session — hybrid state is per request")

 16185 → Red


   521 → Red,
 11401 → Red, Blue


   521 → Red, Blue,
  6941 → Red, Blue, Green


     7 → Red, Blue, Green
[6 tokens, eos] Red, Blue, Green


trace 1 reproduced after an unrelated session — hybrid state is per request


## 12 · Resources and where this connects

Numbers below are measured on *this* host and kernel, not claims for other machines. The
evaluation project behind this notebook carries the same implementation further, with its
own acceptance gates per lane:

- **Reusable library and runtime** — the `lfm2` / `klfm2` modules ported to
  SKaiNET-transformers (the code in this notebook is a cell-by-cell transcription of them).
- **Compiled execution** — StableHLO export + IREE: the static 7-token prefill passed
  compiled CPU replay on the VENTUNO Q device; compiled *cached decode* and the full GPU
  lane have explicit open gates.
- **Cartridge packaging** — a measured, unsigned CPU cartridge pack exists behind the
  SKaiNET cartridge blueprint spec; signing and conformance remain open.

None of those gates affect what this notebook demonstrated: the eager FP32 hybrid model,
its state contract, and byte-exact reference traces on the pinned checkpoint.

In [21]:
val runtime = Runtime.getRuntime()
System.gc()
val usedHeap = (runtime.totalMemory() - runtime.freeMemory()) / (1 shl 20)
println("JVM heap in use : %,d MB (max %,d MB)".format(usedHeap, runtime.maxMemory() / (1 shl 20)))
println("parameters      : %,d (FP32 in memory: %.2f GB)".format(parameterCount, parameterCount * 4.0 / (1 shl 30)))
println("checkpoint load : %.1f s".format(loadMillis / 1000.0))
println("context bound   : ${dims.contextLimit} tokens (tested allocation, not the 128k claim)")

JVM heap in use : 3,722 MB (max 10,240 MB)
parameters      : 354,483,968 (FP32 in memory: 1.32 GB)


checkpoint load : 0.2 s
context bound   : 256 tokens (tested allocation, not the 128k claim)


---
*Model:* [`LiquidAI/LFM2.5-350M`](https://huggingface.co/LiquidAI/LFM2.5-350M) @
`9e6c6ccf47cd318696e137d381a7ded8fe4df09f`, weights under the **LFM Open License v1.0**
(separate download, byte-pinned above). *Code:* MIT, on **SKaiNET 0.62.0**. The Unicode
`Split` tokenizer adaptation reproduces a fix contributed to SKaiNET core; once a release
containing it is adopted here, the in-memory adaptation cell can be retired.*